# ProGen2 Protein Sequence Generation and Scoring

This notebook demonstrates both functions of the ProGen2 tool. In the first section, we use `run_progen2_sample` to generate protein sequences from amino acid prompts using autoregressive sampling. In the second section, we use `run_progen2_score` to compute log-likelihood and perplexity metrics that quantify how natural a given sequence appears to the model, enabling ranking and filtering of design candidates.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("progen2")
display_overview("progen2")
display_docs_section("progen2", "Background")

# ProGen2

> ProGen2 is Salesforce's autoregressive protein language model for de novo protein sequence generation and scoring. Unlike masked language models (ESM2/ESM3) that predict masked positions bidirectionally, ProGen2 generates proteins left-to-right from a prompt and provides autoregressive likelihood scoring. The tool supports local GPU execution via a standalone venv .

## Background

Proteins are linear chains of [amino acids](https://en.wikipedia.org/wiki/Amino_acid) that fold into 3D structures to carry out biological functions. The amino acid sequence ([primary structure](https://en.wikipedia.org/wiki/Protein_structure#Primary_structure)) largely determines the protein's fold and function. Natural proteins occupy a tiny fraction of theoretically possible sequence space -- most random sequences do not fold or function.

[Autoregressive](https://en.wikipedia.org/wiki/Autoregressive_model) protein language models like ProGen2 learn the statistical patterns of natural protein sequences from large databases ([UniRef](https://www.uniprot.org/help/uniref), BFD, [OAS](https://opig.stats.ox.ac.uk/webapps/oas/)). By training to predict each amino acid given the preceding context, the model implicitly captures:

* **Local motifs**: secondary structure preferences, active site patterns
* **Long-range dependencies**: distal residue co-evolution, domain boundaries
* **Family-specific grammar**: sequence patterns characteristic of particular protein families

This learned distribution enables two key applications: **generation** (sampling new sequences that follow natural protein statistics) and **scoring** (evaluating how "protein-like" a given sequence is under the model). Lower perplexity indicates a sequence is more consistent with the model's learned distribution of natural proteins.

## Available tools

In [2]:
display_available_tools("progen2")

- **`run_progen2_sample()`** — Sample protein sequences using ProGen2 language model
- **`run_progen2_score()`** — Score protein sequences using ProGen2 language model

### `run_progen2_sample`

ProGen2 generates protein sequences autoregressively from a prompt, extending the sequence one residue at a time according to its learned distribution over natural proteins. Available checkpoints range from 151M to 6B parameters, with a specialized OAS variant for antibody sequences. The `temperature` and `top_p` parameters control sampling diversity: lower temperature produces more conservative, high-confidence extensions while higher values explore more of sequence space.

In [3]:
from proto_tools import (
    ProGen2SampleInput,
    ProGen2SampleConfig,
    run_progen2_sample,
)

In [4]:
# Display docs
display_api_reference("progen2", "input", "run_progen2_sample")

# Create a ProGen2 input with an N-terminal amino acid prompt
inputs = ProGen2SampleInput(prompts=["MKTAYIAKQRQISF"])

**Input** — `CausalModelSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `prompts` | `List[string]` | required | Prompt sequences to condition generation on. Can be provided as a single string or a list of strings. |

In [5]:
# Display config docs
display_api_reference("progen2", "config", "run_progen2_sample")

# Configure sampling parameters | see docs above for all fields
config = ProGen2SampleConfig(
    model_checkpoint="progen2-large",
    max_length=120,
    temperature=0.2,
    top_p=0.95,
    top_k=0,
    truncate_at_stop=True,
    strip_special_tokens=True,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ProGen2SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `enum` | `progen2-large` | ProGen2 model checkpoint to use. Options: Available options: `progen2-small`, `progen2-medium`, `progen2-oas`, `progen2-large`, `progen2-BFD90`, `progen2-xlarge` |
| `top_k` | `integer` | `0` | Limits sampling to the top-k most probable tokens at each generation step. Set to 0 to disable (use top\_p only). |
| `max_length` | `integer` | `256` | Maximum total sequence length including prompt. Generation stops when this length is reached or a stop token is encountered. Must be at least 1. Default: 256. |
| `truncate_at_stop` | `boolean` | `True` | Whether to truncate generated sequences at the first stop token ('1' or '2'). If `True`, returns clean protein sequences. Default: `True`. |
| `strip_special_tokens` | `boolean` | `True` | Whether to remove the ProGen2 start and stop tokens ('1' or '2') from the output. If `True`, returns clean amino acid sequences. Default: `True`. |
| `return_logits` | `boolean` | `False` | Whether to include per-position logits in the output. When `True`, returns logits for each sequence. When `False`, only returns metrics (saves memory and serialization time). Default: `False`. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `prepend_prompt` | `boolean` | `True` | Whether to include the input prompt at the start of each generated sequence. Default: `True`. |
| `temperature` | `number` | `0.2` | Sampling temperature. Lower values produce more deterministic sequences. Default: 0.2 (following ProGen2 defaults). |
| `top_p` | `number` | `0.95` | Nucleus sampling threshold. Default: 0.95 (following ProGen2 recommendations). |
| `batch_size` | `integer` | `1` | Number of sequences to process simultaneously. |
| `local_path` | `string` |  | Optional path to local model weights directory. If provided, loads model from local filesystem instead of downloading from HuggingFace. Useful for offline inference or custom model versions. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run the model on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the sampling function
sample_result = run_progen2_sample(inputs, config)

Running run_progen2_sample [00:00]

In [7]:
# Display output docs
display_api_reference("progen2", "output", "run_progen2_sample")

# Show the prompt and the generated continuation
prompt = inputs.prompts[0]
generated = sample_result.sequences[0]
print(f"Prompt:       {prompt}")
print(f"Generated:    {generated[:80]}..." if len(generated) > 80 else f"Generated:    {generated}")
print(f"Total length: {len(generated)} residues")

**Output** — `ProGen2SampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `logits` | `array` |  | Per-position logits for each generated sequence (shape: \[num\_sequences, generated\_len, vocab\_size]). |
| `sequences` | `List[string]` | required | Generated protein sequences. |

Prompt:       MKTAYIAKQRQISF
Generated:    MKTAYIAKQRQISFKEALKLLEESGIASLPVVDENDKLLGLLSVTDVLRHEFVDELIEEGETVDAYIHATPTTLTPQMPL...
Total length: 120 residues


### `run_progen2_score`

ProGen2 scores sequences by computing the autoregressive log-likelihood at each position, measuring how predictable each residue is given its preceding context. The resulting perplexity summarizes sequence naturalness: lower perplexity indicates the sequence more closely matches the statistical patterns of natural proteins in the training data. This is useful for ranking designed candidates, filtering out sequences unlikely to fold or function, or comparing wild-type and mutant variants.

In [8]:
from proto_tools import (
    ProGen2ScoringInput,
    ProGen2ScoringConfig,
    run_progen2_score,
)

In [9]:
# Display docs
display_api_reference("progen2", "input", "run_progen2_score")

# Score two protein sequences
score_inputs = ProGen2ScoringInput(sequences=["MVLSPADKTN", "MKTLLILAVVAA"])

**Input** — `CausalModelScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Sequences to score. Can be provided as a single string or a list of strings. |

In [10]:
# Display config docs
display_api_reference("progen2", "config", "run_progen2_score")

# Configure scoring | see docs above for all fields
score_config = ProGen2ScoringConfig(
    batch_size=2,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ProGen2ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `enum` | `progen2-large` | ProGen2 model checkpoint to use. Options include `"progen2-small"` (151M), `"progen2-medium"` (754M), `"progen2-oas"` (754M, antibody-specific), `"progen2-large"` (2B), `"progen2-BFD90"` (2B), `"progen2-xlarge"` (6B). Available options: `progen2-small`, `progen2-medium`, `progen2-oas`, `progen2-large`, `progen2-BFD90`, `progen2-xlarge` |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `batch_size` | `integer` | `1` | Number of sequences to process simultaneously on GPU. Larger batches improve throughput but use more GPU memory. |
| `return_logits` | `boolean` | `False` | Whether to include per-position logits in the output. |
| `local_path` | `string` |  | Optional path to local model weights directory. If provided, loads model from local filesystem instead of downloading from HuggingFace. Default: `None`. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run the model on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [11]:
# Run the scoring function
score_result = run_progen2_score(score_inputs, score_config)

Running run_progen2_score [00:00]

In [12]:
# Display output docs
display_api_reference("progen2", "output", "run_progen2_score")

# Show scoring results for each input sequence
for i, seq in enumerate(score_inputs.sequences):
    score = score_result.scores[i]
    print(f"Sequence {i + 1}: {seq}")
    print(f"    Log-likelihood:     {score.log_likelihood:.3f}")
    print(f"    Avg log-likelihood: {score.avg_log_likelihood:.3f}")
    print(f"    Perplexity:         {score.perplexity:.3f}")

**Output** — `CausalModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `List[CausalModelScoringMetrics]` | required | List of scoring outputs, one per input sequence. Each entry contains metrics (log\_likelihood, avg\_log\_likelihood, perplexity) and optional per-position logits. |
| `metrics` | `Dict[string, number]` |  | Dictionary of scalar scoring metrics. |
| `logits` | `array` |  | Optional per-position logits array. |
| `vocab` | `array` |  | Optional token ordering for logits; logits\[:, j] corresponds to vocab\[j]. |
| `per_position_metrics` | `Dict[string, List[number]]` |  | Optional per-position scoring metrics, keyed by metric name. Each value is a list of length equal to the input sequence, with `None` at positions where that metric is not available. |

Sequence 1: MVLSPADKTN
    Log-likelihood:     -26.635
    Avg log-likelihood: -2.664
    Perplexity:         14.346
Sequence 2: MKTLLILAVVAA
    Log-likelihood:     -24.403
    Avg log-likelihood: -2.034
    Perplexity:         7.641


## Export Results

Generated sequences and scoring metrics can be saved to standard file formats for downstream use. Sampling results support FASTA, TXT, and JSON export. Scoring results support CSV and JSON export.

In [13]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sample_result.export(name="progen2_sequences", export_path=output_dir, file_format="fasta")
print(f"Generated sequences exported to {output_dir / 'progen2_sequences.fasta'}")

score_result.export(name="progen2_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'progen2_scores.csv'}")

Generated sequences exported to example_output/progen2_sequences.fasta
Scores exported to example_output/progen2_scores.csv
